# 完整配音流水线（GPU）

依次执行：TIGER-DnR 分离原片 → NVIDIA RE-USE 修复演员干声 → 排 Reaper 工程。
顶部菜单选 **运行时 → 更改运行时类型 → T4 GPU**（免费）。

注意：RE-USE 是 NVIDIA 非商用许可（NSCLv1），毕业论文/非商业用途没问题。

In [ ]:
import shutil, subprocess, sys
shutil.rmtree('/content/MyDubbingMixingLab', ignore_errors=True)
!git clone -q https://github.com/Fectxd/MyDubbingMixingLab.git
%cd /content/MyDubbingMixingLab
!pip install -q einops pyyaml soundfile reathon

# mamba-ssm 在 PyPI 只有源码包；直接从官方 GitHub Releases 下载与
# 当前 torch/CUDA/Python 匹配的预编译轮子（比源码编译快且不会失败）
import torch
print('python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.cuda.is_available())
tv = '.'.join(torch.__version__.split('+')[0].split('.')[:2])
py = f'cp{sys.version_info.major}{sys.version_info.minor}'
cu = 'cu13' if (torch.version.cuda or '').startswith('13') else 'cu12'
url = f'https://github.com/state-spaces/mamba/releases/download/v2.3.2.post1/mamba_ssm-2.3.2.post1%2B{cu}torch{tv}cxx11abiTRUE-{py}-{py}-linux_x86_64.whl'
print('wheel:', url.split('/')[-1])
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', url])
if r.returncode != 0:
    print('这个 wheel 不存在，请把上面的 torch 版本发给我')
else:
    import mamba_ssm
    print('mamba_ssm OK')

In [ ]:
import os
from google.colab import files
os.makedirs('test', exist_ok=True)
print('请上传：原片（原片.mp4）+ 5 条演员干声 wav（可多选）')
up = files.upload()
for name, data in up.items():
    with open(os.path.join('test', name), 'wb') as f:
        f.write(data)
print('已保存：', list(up))

In [ ]:
!python separate.py --input test/原片.mp4 --device auto

In [ ]:
!python enhance.py --inputs test/*.wav --outdir work/enhanced

In [ ]:
!python assemble_rpp.py --actors test

In [ ]:
!zip -rq output.zip work
from google.colab import files
files.download('output.zip')
print('下载完成后，解压 output.zip 即可得到全部结果')